# 01 · Prompt와 모델

**목표**: Gen AI Hub SDK로 LLM을 호출하고, message role과 system instruction이 출력에 미치는 영향을 비교합니다.

LLM은 다음 토큰을 예측하는 모델이며, 우리가 보내는 입력 전체를 prompt라고 합니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Prompt Chatbot (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 system prompt, 모델, 실행 방식을 바꾸며 대화를 이어가세요. ReAct Agent는 필요할 때 읽기 전용 계산 도구를 호출한 뒤, 그 관찰값을 답변에 반영합니다. 대화 history는 현재 브라우저 session에만 유지됩니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>질문을 입력하고 답변을 확인합니다.</li>
<li>⚙️ 설정에서 일반 Chat 또는 ReAct Agent를 선택합니다. ReAct Agent에서는 <code>15% 할인 뒤 3개 가격은?</code>처럼 계산이 필요한 질문을 입력해 보세요.</li>
<li>이 UI는 업무 상태를 변경하지 않습니다.</li>
</ol>
  </div>
</div>


In [2]:
from pathlib import Path
import json
from time import perf_counter
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lab_core import run_adaptive_react_agent, run_chat, run_chat_stream, run_react_agent
from src.notebook_ui import build_chat_ui
chat_ui = build_chat_ui(run_chat, chat_stream_runner=run_chat_stream, react_agent_runner=run_react_agent, adaptive_react_agent_runner=run_adaptive_react_agent)
chat_ui.launch(inline=True, share=False, server_name='127.0.0.1', show_error=True, width='100%', height=720)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">메시지 role과 준비</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li><code>system</code>은 모델의 역할, 규칙, 언어, 출력 형식을 정합니다.</li>
<li><code>user</code>는 사용자의 질문 또는 요청입니다.</li>
<li><code>assistant</code>는 이전 모델 응답으로, 대화 history에 사용합니다.</li>
</ul>
<p style="margin: 7px 0;">아래 코드는 루트 <code>.env</code>의 <code>LLM_MODEL</code>을 사용합니다. UI를 실행한 뒤에는 선택 심화 자료로 코드 기반 비교를 확인하세요.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


In [3]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path[:0] = [str(project_root / '1_gen_ai_hub' / 'src'), str(project_root / '2_gen_ai_hub_llm')]
from src.prompt_model.lesson_support.llm import chat_text
from src.prompt_model.lesson_support.config import load_settings
from src.prompt_model.lab_core import compare_prompts, run_chat
settings = load_settings()
print(f'LLM model: {settings.llm_model}')

LLM model: gpt-5.6-luna


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 prompt 비교하기</h2>
<p style="margin: 7px 0;">이 아래 셀들은 UI와 같은 LLM 호출을 공개 코어 <code>compare_prompts()</code>로 확인하는 심화 자료입니다. 기본 실습에는 실행할 필요가 없습니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">기본 chat 응답</h3>
<p style="margin: 7px 0;">같은 질문을 기본 조건과 표 형식 system 조건으로 함께 호출하고, 먼저 기본 응답을 확인합니다.</p>
  </div>
</div>


In [23]:
question = 'LLM의 기본 아키텍처를 설명해줘.'
comparison_system_prompt = """너는 친절한 선생님이다.
초등학생도 이해할 수 있게 쉬운 한국어로 설명한다.
첫 줄에는 한 문장 요약을 쓴다.
그 아래에 핵심 내용을 bullet 3개로 정리한다."""

# comparison_system_prompt를 직접 바꿔 원하는 제약을 실험하세요.
comparison = compare_prompts(question, settings, system_prompt=comparison_system_prompt)
print(comparison['baseline'])

LLM(Large Language Model)의 기본 아키텍처는 대부분 **Transformer**를 기반으로 합니다. 텍스트를 입력받아 다음에 올 토큰의 확률을 계산하고, 이를 반복해 문장을 생성합니다.

## 1. 전체 흐름

```text
텍스트 입력
   ↓
토큰화(Tokenization)
   ↓
토큰 임베딩 + 위치 정보
   ↓
Transformer 블록 여러 층
   ↓
출력 확률 계산
   ↓
다음 토큰 선택
   ↓
반복하여 문장 생성
```

## 2. 토큰화

문장을 모델이 처리할 수 있는 작은 단위인 **토큰(token)** 으로 나눕니다.

예를 들어:

```text
"나는 학교에 간다"
→ ["나는", "학교", "에", "간다"]
```

실제 LLM은 단어 전체뿐 아니라 단어 일부, 기호, 공백 등을 토큰으로 사용할 수 있습니다.

## 3. 임베딩

각 토큰을 고정된 길이의 숫자 벡터로 변환합니다.

```text
토큰 ID → 임베딩 벡터
```

이 벡터에는 단어의 의미와 사용 패턴이 학습 과정에서 반영됩니다.

또한 모델은 토큰의 순서를 알아야 하므로 **위치 정보(position information)** 를 추가합니다. Transformer는 기본적으로 순서를 직접 알 수 없기 때문입니다.

## 4. Transformer 블록

LLM은 Transformer 블록을 여러 층 쌓은 구조입니다. 각 블록은 주로 다음 요소로 구성됩니다.

### 4.1 Self-Attention

문장 안의 각 토큰이 다른 토큰들을 얼마나 참고해야 하는지 계산합니다.

예를 들어:

```text
"철수가 사과를 먹었다. 그는 맛있다고 했다."
```

여기서 “그는”이 “철수”를 가리킨다는 관계를 파악하려면 서로 멀리 떨어진 토큰 간의 연결을 살펴봐야 합니다. Self-Attention은 이런 관계를 계산합니다.

일반적으로 각 토큰에서 다음 세 가지 벡터를 만듭니다.

- Query: 무엇을 찾는가
- Key: 어떤 정보를 가지고 있

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">system prompt로 제약하기</h3>
<p style="margin: 7px 0;">같은 코어 호출에서 반환된 표 형식 결과를 확인합니다. 제목·표·bullet처럼 검증하기 쉬운 제약은 기본 응답과 차이를 더 분명하게 만듭니다.</p>
  </div>
</div>


In [24]:
print(comparison['constrained'])

LLM은 문장을 숫자로 바꾸고, 트랜스포머로 문맥을 이해한 뒤, 다음에 올 단어를 하나씩 예측하는 인공지능입니다.

- **입력 변환**: 문장을 작은 단위인 **토큰**으로 나누고, 각 토큰을 숫자 벡터인 **임베딩**으로 바꿉니다. 문장에서 단어의 순서도 알 수 있도록 위치 정보도 더합니다.
- **문맥 이해**: 여러 개의 **트랜스포머 층**이 토큰 사이의 관계를 살핍니다. 특히 **어텐션**은 “현재 단어가 다른 어떤 단어를 중요하게 봐야 하는가?”를 계산해 문맥을 이해하게 합니다.
- **다음 토큰 예측**: 마지막에는 지금까지의 내용을 바탕으로 다음에 올 토큰의 확률을 계산합니다. 가장 알맞은 토큰을 고르고, 이 과정을 반복해 문장을 완성합니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 2</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">message 배열을 직접 구성하기</h2>
<p style="margin: 7px 0;">LLM API에는 하나의 문자열이 아니라 역할이 붙은 메시지 배열을 보냅니다. 아래에서 <code>system</code> 규칙, 이전 <code>user</code> 질문, 이전 <code>assistant</code> 답변, 새 <code>user</code> 질문의 순서를 확인하고 실행하세요.</p>
  </div>
</div>


In [25]:
messages = [
    {"role": "system", "content": "너는 친절한 교육 보조자다. 한국어로 간결하게 답한다."},
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
    {"role": "user", "content": "방금 답을 표로 정리해줘."},
]
for index, item in enumerate(messages, start=1):
    print(f'{index}. {item["role"]}: {item["content"]}')

reply = chat_text(settings, messages, settings.llm_model)
print()
print('응답:')
print(reply)

1. system: 너는 친절한 교육 보조자다. 한국어로 간결하게 답한다.
2. user: LLM의 장점을 두 가지 알려줘.
3. assistant: 자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다.
4. user: 방금 답을 표로 정리해줘.

응답:
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 답변할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 요약, 번역, 작성, 교정 등 여러 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 3</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">대화 history가 맥락을 전달하는 방식</h2>
<p style="margin: 7px 0;">두 번째 질문만 보면 무엇을 표로 바꾸라는지 알 수 없습니다. 같은 질문을 history 없이, 그리고 history와 함께 보내 결과를 비교해 보세요.</p>
<p style="margin: 7px 0;"><strong>관찰:</strong> 코어는 최근 12개 메시지와 총 12,000자 안에서만 history를 보존합니다. 긴 대화에서는 중요한 맥락을 요약하거나 문서 검색으로 보완해야 합니다.</p>
  </div>
</div>


In [26]:
follow_up = '방금 답을 표로 정리해줘.'
history = [
    {"role": "user", "content": "LLM의 장점을 두 가지 알려줘."},
    {"role": "assistant", "content": "자연어를 이해하고 다양한 텍스트 작업을 도울 수 있습니다."},
]
without_history = run_chat(follow_up, settings)
with_history = run_chat(follow_up, settings, history=history)
print('[history 없음]')
print(without_history['reply'])
print()
print('[history 포함]')
print(with_history['reply'])

[history 없음]
표로 정리할 답변 내용이 현재 대화에 보이지 않습니다. 방금 답변을 붙여 넣어 주시면 바로 표로 정리해 드리겠습니다.

[history 포함]
| 장점 | 설명 |
|---|---|
| 자연어 이해 | 사람의 질문과 문장을 이해하고 자연스럽게 응답할 수 있습니다. |
| 다양한 텍스트 작업 지원 | 요약, 번역, 작성, 편집 등 여러 텍스트 작업을 도울 수 있습니다. |


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  실습 4</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">프롬프트로 JSON 형식 요청하고 검증하기</h2>
<p style="margin: 7px 0;">형식을 요청해도 항상 파싱 가능한 JSON이 반환된다고 보장할 수는 없습니다. 실행 결과가 파싱에 성공하는지 확인하고, 실패했다면 모델이 반환한 원문을 살펴보세요.</p>
  </div>
</div>


In [27]:
json_prompt = (
    '너는 교육 보조자다. 반드시 JSON 객체만 반환한다. Markdown 코드 블록이나 설명을 덧붙이지 않는다. '
    '키는 summary(문자열), keywords(문자열 배열)만 사용한다.'
)
json_outcome = run_chat(
    'LLM을 한 문장으로 요약하고 핵심 키워드 세 개를 추출해줘.',
    settings,
    system_prompt=json_prompt,
)
try:
    parsed = json.loads(json_outcome['reply'])
    print('JSON 파싱 성공:', parsed)
except json.JSONDecodeError:
    print('JSON 파싱 실패. 반환 원문을 확인하세요:')
    print(json_outcome['reply'])

JSON 파싱 성공: {'summary': 'LLM은 방대한 텍스트 데이터를 학습해 인간의 언어를 이해하고 생성하는 인공지능 모델이다.', 'keywords': ['대규모 언어 모델', '자연어 처리', '생성형 인공지능']}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 실험</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">동일 질문을 모델별로 비교하기</h2>
<p style="margin: 7px 0;">현재 모델과 AI Core에 배포된 다른 모델을 같은 질문으로 호출해 보세요. 응답 시간뿐 아니라 정확성, 지시 준수, 출력 형식을 함께 기록합니다. 배포되지 않은 모델명은 호출할 수 없습니다.</p>
  </div>
</div>


In [28]:
models_to_compare = [settings.llm_model]
# AI Core에 RUNNING 상태로 배포된 모델이 있다면 아래처럼 추가합니다.
models_to_compare.append('gpt-4.1')

comparison_question = 'SAP에서 2026년도에 출시한 제품은?'
for model_name in models_to_compare:
    started_at = perf_counter()
    outcome = run_chat(comparison_question, settings, model=model_name)
    elapsed = perf_counter() - started_at
    print(f'[{model_name}] {elapsed:.1f}초')
    print(outcome['reply'])
    print('='*100 + '\n')

[gpt-5.6-luna] 6.9초
“2026년에 출시한 SAP 제품”을 **완전히 새로운 제품**으로 볼지, 기존 제품의 **2026년 릴리스**로 볼지에 따라 다릅니다.

현재 공개적으로 확인 가능한 범위에서 대표적인 2026년 SAP 릴리스는 다음과 같습니다.

- **SAP S/4HANA Cloud Public Edition 2602** — 2026년 상반기 릴리스
- **SAP S/4HANA Cloud Public Edition 2608** — 2026년 하반기 릴리스
- **SAP Business Technology Platform(BTP)의 2026년 기능 업데이트** — 서비스별로 분기 또는 수시 제공
- **SAP SuccessFactors, SAP Ariba, SAP Concur 등 클라우드 제품의 2026년 릴리스** — 제품별 업데이트 방식

다만 이는 대부분 **신규 제품 출시가 아니라 기존 클라우드 제품의 버전·기능 업데이트**입니다. SAP가 2026년에 완전히 새로 출시한 제품의 공식 목록은 SAP의 발표 자료와 제품별 릴리스 문서를 기준으로 확인해야 하며, SAP는 보통 다음 사이트에서 확인할 수 있습니다.

- **SAP Product Availability Matrix(PAM)**
- **SAP What’s New Viewer**
- SAP News Center의 2026년 제품 발표 자료

참고로 **SAP Business Data Cloud**와 **Joule**, **SAP Cloud ERP**는 2026년 제품이 아니라 그 이전에 발표·출시된 제품입니다.

[gpt-4.1] 2.6초
현재까지의 정보(2024년 6월 기준)로는 SAP에서 2026년도에 출시 예정이거나 출시된 제품에 대한 공식 발표가 없습니다. SAP는 주로 새로운 클라우드 솔루션, S/4HANA, 인공지능(AI) 통합 기능, 비즈니스 네트워크 등의 신제품 및 기능 개선을 지속적으로 발표하고 있습니다. 그러나 2026년에 특정 제품이 출시된다는 공식 뉴스나

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🤖  ReAct 코드 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct Agent의 답변 생성 과정</h2>
<p style="margin: 7px 0;">계산 도구 실행 뒤 Agent가 최종 답변을 만드는 과정을 확인합니다. <code>Thinking</code>은 상세 내부 추론이 아니라 공개 가능한 요약만 표시합니다.</p>
<p style="margin: 7px 0;">참고: <a href="https://huggingface.co/datasets/qrk-labs/akeel-thought-injection-50k-augmented/viewer/default/train?p=1" target="_blank" rel="noopener noreferrer">reasoning tags 예시 (Hugging Face)</a></p>
  </div>
</div>


In [4]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI
from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment

react_system_prompt = '''
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
'''.strip()

@tool
def calculate(expression: str) -> str:
    """사칙연산과 괄호가 있는 수식을 계산합니다."""
    return str(_calculator(expression))

apply_ai_core_environment(settings)
model = ChatOpenAI(model=settings.llm_model, api_key="EMPTY", client=OpenAI().chat.completions)
react_agent = create_agent(model=model, tools=[calculate], system_prompt=react_system_prompt)

react_question = '정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?'
max_react_loops = 5  # 최대 도구 실행 횟수; 답이 완성되면 더 일찍 종료
result = react_agent.invoke(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
)

for message in result["messages"]:
    if message.type == "ai" and message.content:
        print(message.content)


Question: 정가 12,000원인 상품을 15% 할인해 3개 구매하면 총 얼마인가요?  
Thought: 할인된 단가를 계산한 뒤 3개를 곱하겠습니다.  
Action: calculate  
Action Input: {"expression":"12000*(1-15/100)*3"}
정가 12,000원에서 15% 할인하면 개당 **10,200원**입니다.

3개 구매 시 총액은 **30,600원**입니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #7b1fa2; background: #f8f1fb; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #7b1fa2; letter-spacing: .02em;">🧪 Reasoning Tag 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">ReAct 실행 기록을 태그 흐름으로 읽기</h2>
<p style="margin: 7px 0;">아래 셀의 질문을 바꿔 실행해 보세요. <code>&lt;thinking&gt;</code>은 상세 내부 추론이 아니라 다음 행동의 공개 가능한 짧은 근거입니다.</p>
<p style="margin: 7px 0;">실행 결과의 <code>&lt;thinking&gt;</code>, <code>&lt;evaluation&gt;</code>, <code>&lt;answer&gt;</code>를 비교해 보세요. 실제 도구 호출과 반환값은 trace의 <code>action</code>, <code>observation</code>에서 확인합니다.</p>
  </div>
</div>


In [15]:
import json

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from gen_ai_hub.proxy.native.openai import OpenAI

from src.prompt_model.lab_core import _calculator
from src.prompt_model.lesson_support.ai_core import apply_ai_core_environment


react_system_prompt = '''
너는 Joule 구매·회계·인사·제조원가 지원 ReAct Agent다.
Plan → Thought → Action → Observation → Evaluation → Final Answer 순서를 따른다.

절대로 Question:, Thought:, Action:, Action Input:, Observation:, Final Answer:
같은 텍스트 형식을 사용하지 않는다. XML 태그 형식만 사용한다.

태그 역할:
- <plan>: 해결할 작업의 짧은 계획
- <assumptions>: 답변에 사용한 전제
- <thinking>: 다음 행동을 선택하는 공개 가능한 짧은 근거
- <tool_selection>: 사용할 도구와 선택 이유
- <evaluation>: 도구 결과의 정확성·충분성 점검
- <reflection>: 오류·누락·추가 확인 필요 여부
- <answer>: 최종 사용자 답변

상품, 재고, 예산, 회계 정책, 직원·결재선, 제조원가, 생산능력,
외부 견적, 품질 요건은 반드시 해당 도구로 확인한다.
계산은 calculate 도구로 검증하고, 도구 반환값만 사실로 사용한다.

첫 도구 호출 전에는 다음 태그를 작성한다.

<plan>...</plan>
<assumptions>...</assumptions>
<thinking>...</thinking>
<tool_selection>...</tool_selection>

도구 결과를 받은 뒤 최종 응답은 다음 태그를 작성한다.

<thinking>...</thinking>
<evaluation>...</evaluation>
<reflection>...</reflection>
<answer>...</answer>

각 태그는 한두 문장으로 간결하게 작성한다.
상세 내부 추론이나 장문의 풀이 과정은 작성하지 않는다.
'''.strip()

@tool
def get_facility_staffing_request(site: str) -> str:
    """설비팀: 설비 운영 현황과 필요한 인력 충원 요청을 조회합니다."""
    requests = {
        "PLANT-SEOUL": {
            "request_id": "FAC-REQ-2026-041",
            "site": "PLANT-SEOUL",
            "equipment_area": "조립 라인 설비보전",
            "current_headcount": 4,
            "required_headcount": 6,
            "requested_headcount": 2,
            "position": "설비보전 Technician",
            "reason": "예방정비와 긴급 고장 대응 인력이 부족합니다.",
            "required_start_date": "2026-12-01",
        }
    }
    return json.dumps(
        requests.get(site, {"error": "설비팀 요청을 찾을 수 없습니다."}),
        ensure_ascii=False,
    )


@tool
def prepare_hr_recruitment_request(
    facility_request_id: str,
    position: str,
    headcount: int,
    required_start_date: str,
) -> str:
    """인사팀: 설비팀 인력 요청을 바탕으로 채용요청 초안을 작성합니다."""
    return json.dumps({
        "status": "DRAFT",
        "hr_request_id": "HR-REQ-2026-118",
        "facility_request_id": facility_request_id,
        "position": position,
        "requested_headcount": headcount,
        "required_start_date": required_start_date,
        "estimated_recruiting_days": 30,
        "approval_required": ["설비팀장", "HR Business Partner"],
    }, ensure_ascii=False)


@tool
def get_manufacturing_plan_impact(hr_request_id: str) -> str:
    """제조기획: 채용 요청이 생산계획과 인력 투입 일정에 미치는 영향을 확인합니다."""
    return json.dumps({
        "hr_request_id": hr_request_id,
        "production_plan": "2027년 1분기 조립라인 생산량 20% 증산",
        "recommended_onboarding_date": "2026-12-01",
        "staffing_priority": "HIGH",
        "temporary_workaround": "채용 완료 전까지 외주 정비 인력 1명 투입",
    }, ensure_ascii=False)


@tool
def get_hr_recruitment_sop(request_type: str) -> str:
    """인사팀의 채용 요청 SOP와 필수 검토 항목을 조회합니다."""
    sops = {
        "facility_staffing": {
            "sop_name": "설비 인력 충원 및 채용 요청 SOP",
            "process_sequence": [
                "설비팀 인력 충원 요청 확인",
                "인사팀 채용요청 초안 작성",
                "제조기획팀 생산계획 영향 검토",
                "설비팀장 및 HR Business Partner 결재",
            ],
            "required_fields": [
                "직무",
                "요청 인원",
                "충원 사유",
                "입사 필요일",
                "생산계획 영향",
            ],
            "restriction": "채용 공고 등록 전에는 승인된 채용요청이 필요합니다.",
        }
    }
    return json.dumps(
        sops.get(request_type, {"error": "해당 채용 SOP를 찾을 수 없습니다."}),
        ensure_ascii=False,
    )
    
react_agent = create_agent(
    model=model,
    tools=[
        get_hr_recruitment_sop,
        get_facility_staffing_request,
        prepare_hr_recruitment_request,
        get_manufacturing_plan_impact,
    ],
    system_prompt=react_system_prompt,
)

react_question = '''
PLANT-SEOUL 설비팀에서 설비보전 Technician 인력 충원이 필요합니다.

사내 인사 채용 SOP를 확인해 요청을 검토하고,
채용요청 초안을 준비한 뒤 생산계획 영향까지 확인해 주세요.
실제 채용 공고는 등록하지 않습니다.
'''

max_react_loops = 8

for update in react_agent.stream(
    {"messages": [{"role": "user", "content": react_question}]},
    {"recursion_limit": max_react_loops * 2 + 2},
    stream_mode="updates",
):
    for node_name, state in update.items():
        for message in state.get("messages", []):
            if message.type == "ai" and message.content:
                print(f"\n[{node_name}]")
                print(message.content)

            if message.type == "ai" and message.tool_calls:
                for call in message.tool_calls:
                    print(f"<action>{call['name']}({call['args']})</action>")

            if message.type == "tool":
                print(f"<observation>{message.content}</observation>")


[model]
<plan>채용 SOP와 PLANT-SEOUL 설비팀의 충원 요청을 확인한 뒤, 확인된 요청 정보를 바탕으로 채용요청 초안을 작성하고 생산계획 영향을 검토하겠습니다.</plan>
<assumptions>사용자가 요청한 범위는 검토·초안 작성·생산계획 영향 확인이며, 실제 채용 공고 등록은 수행하지 않습니다.</assumptions>
<thinking>SOP와 설비팀 요청은 서로 독립적으로 조회할 수 있으므로 먼저 병렬 확인하겠습니다.</thinking>
<tool_selection>get_hr_recruitment_sop와 get_facility_staffing_request를 병렬 호출합니다.</tool_selection>

<action>get_hr_recruitment_sop({'request_type': '설비보전 Technician 인력 충원'})</action>
<action>get_facility_staffing_request({'site': 'PLANT-SEOUL'})</action>
<observation>{"request_id": "FAC-REQ-2026-041", "site": "PLANT-SEOUL", "equipment_area": "조립 라인 설비보전", "current_headcount": 4, "required_headcount": 6, "requested_headcount": 2, "position": "설비보전 Technician", "reason": "예방정비와 긴급 고장 대응 인력이 부족합니다.", "required_start_date": "2026-12-01"}</observation>
<observation>{"error": "해당 채용 SOP를 찾을 수 없습니다."}</observation>

[model]
<thinking>SOP의 최초 조회가 특정 직무명으로는 일치하지 않았지만, 설비팀 요청은 확인되었습니다. 일반 채용요청 SOP를 재조회하는 동안 해당 요청 정보로 초안 준비를 진행할 수 있습니다.<